In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Patch

In [2]:
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUT = ROOT / "261003_betalactone_synteny_ortholog_results"
OUT.mkdir(parents=True, exist_ok=True)

XLSX = "AO090005001236_antiSMASH_20kb.xlsx"
ORTHOLOG_XLSX = "AO090005001236_betalactone_BGC_ortholog.xlsx"
GTF_BY_SPECIES = {
    "Aspergillus_oryzae_RIB40": "aor0-5.gtf",
    "Aspergillus_flavus_NRRL3357": "afl0-1.gtf",
    "Aspergillus_clavatus_NRRL_1": "acl0-1.gtf",
    "Aspergillus_nidulans_FGSC_A4": "and0-2.gtf",
    "Aspergillus_fumigatus_Af293": "afu0-1.gtf",
    "Aspergillus_niger_CBS_513.88": "ang0-2.gtf",
    "Aspergillus_terreus_NIH2624": "atr0-1.gtf",
}
FLANK_BP = 5000
ORTHOLOG_COLUMNS = {
    "Gene ID": "Aspergillus_oryzae_RIB40",
    "Aspergillus flavus NRRL3357": "Aspergillus_flavus_NRRL3357",
    "Aspergillus clavatus NRRL 1": "Aspergillus_clavatus_NRRL_1",
    "Aspergillus nidulans FGSC A4": "Aspergillus_nidulans_FGSC_A4",
    "Aspergillus fumigatus Af293": "Aspergillus_fumigatus_Af293",
    "Aspergillus niger CBS 513.88": "Aspergillus_niger_CBS_513.88",
    "Aspergillus terreus NIH2624": "Aspergillus_terreus_NIH2624",
}

In [4]:
def read_gtf(path):
    d = pd.read_csv(
        path, sep="\t", comment="#", header=None,
        names=["seqname", "source", "feature", "start", "end", "score",
               "strand", "frame", "attributes"],
        usecols=[0, 3, 4, 6, 8], dtype={"seqname": str, "attributes": str},
    )
    d["gene_id"] = d.attributes.str.extract(r'gene_id\s+"([^"]+)"', expand=False)
    d = d[d.gene_id.notna()].copy()
    d[["start", "end"]] = d[["start", "end"]].apply(pd.to_numeric, errors="raise")
    grouped = d.groupby("gene_id", sort=False).agg(
        seqname=("seqname", "first"), start=("start", "min"),
        end=("end", "max"), strand=("strand", "first"),
        n_seqnames=("seqname", "nunique"), n_strands=("strand", "nunique"),
    ).reset_index()
    if grouped.n_seqnames.gt(1).any() or grouped.n_strands.gt(1).any():
        raise ValueError(f"A GTF gene_id spans multiple chromosomes or strands: {path}")
    return grouped.drop(columns=["n_seqnames", "n_strands"])


def transform(pos, row):
    """Orient every locus with the TF at left and the BGC at right.

    x=0 is the TF boundary nearest the BGC. Right-side native loci are flipped.
    """
    if row.gene_bgc_relation == "left_of_bgc":
        return pos - row.end
    if row.gene_bgc_relation == "right_of_bgc":
        return row.start - pos
    raise ValueError(f"Unexpected relation: {row.gene_bgc_relation}")


def prepare(xlsx):
    table = pd.read_excel(xlsx)
    required = {"reference_species", "seqname", "gene_id", "start", "end",
                "nearest_dist", "gene_bgc_relation", "nearest_bgc_id",
                "nearest_bgc_start", "nearest_bgc_end", "nearest_bgc_type"}
    if required - set(table.columns):
        raise ValueError(f"Required columns are missing from the Excel file: {required - set(table.columns)}")
    if table.reference_species.duplicated().any():
        raise ValueError("Duplicate reference_species rows were found")
    if set(table.reference_species) != set(GTF_BY_SPECIES):
        raise ValueError("Excel species names do not match the GTF mapping")
    table["seqname"] = table["seqname"].astype(str)
    table["gene_id"] = table["gene_id"].astype(str)
    table["nearest_bgc_id"] = table["nearest_bgc_id"].astype(str)
    for c in ["start", "end", "nearest_dist", "nearest_bgc_start", "nearest_bgc_end"]:
        table[c] = pd.to_numeric(table[c], errors="raise").astype(int)
    if not table.nearest_bgc_type.astype(str).str.contains("betalactone", case=False).all():
        raise ValueError("Non-betalactone BGCs were found")

    details = []
    summaries = []
    for row in table.itertuples(index=False):
        gtf_name = GTF_BY_SPECIES[row.reference_species]
        gtf_path = ROOT / gtf_name
        has_gtf = gtf_path.exists()
        if not has_gtf:
            raise FileNotFoundError(f"Place {gtf_name} in the input directory: {ROOT}")

        expected_gap = (row.nearest_bgc_start - row.end
                        if row.gene_bgc_relation == "left_of_bgc"
                        else row.start - row.nearest_bgc_end)
        if expected_gap != row.nearest_dist:
            raise ValueError(f"The Excel distance and coordinates do not agree: {row.reference_species}")

        left = min(row.start, row.nearest_bgc_start) - FLANK_BP
        right = max(row.end, row.nearest_bgc_end) + FLANK_BP
        genes = pd.DataFrame(columns=["seqname", "start", "end", "strand", "gene_id"])
        if has_gtf:
            all_genes = read_gtf(gtf_path)
            tf = all_genes[all_genes.gene_id.eq(row.gene_id)]
            if len(tf) != 1 or tf.iloc[0].seqname != row.seqname:
                raise ValueError(f"The TF ID is missing from the GTF or its chromosome differs: {row.gene_id}")
            if int(tf.iloc[0].start) != row.start or int(tf.iloc[0].end) != row.end:
                raise ValueError(f"TF coordinates differ between Excel and GTF: {row.gene_id}")
            genes = all_genes[all_genes.seqname.eq(row.seqname) &
                              all_genes.end.ge(left) & all_genes.start.le(right)].copy()

        bgc_left = min(transform(row.nearest_bgc_start, row),
                       transform(row.nearest_bgc_end, row))
        bgc_right = max(transform(row.nearest_bgc_start, row),
                        transform(row.nearest_bgc_end, row))
        for gene in genes.itertuples(index=False):
            x1, x2 = transform(gene.start, row), transform(gene.end, row)
            x_left, x_right = min(x1, x2), max(x1, x2)
            # Gene touches the antiSMASH area if its interval overlaps it.
            in_bgc = gene.start <= row.nearest_bgc_end and gene.end >= row.nearest_bgc_start
            display_strand = gene.strand if row.gene_bgc_relation == "left_of_bgc" else (
                "+" if gene.strand == "-" else "-" if gene.strand == "+" else gene.strand)
            details.append({
                "reference_species": row.reference_species, "seqname": row.seqname,
                "gene_id": gene.gene_id, "start_1based": int(gene.start),
                "end_1based": int(gene.end), "native_strand": gene.strand,
                "display_strand": display_strand, "display_start_kb": x_left / 1000,
                "display_end_kb": x_right / 1000, "overlaps_antiSMASH_area": bool(in_bgc),
                "is_TF_ortholog": gene.gene_id == row.gene_id,
            })
        summaries.append({
            "reference_species": row.reference_species, "TF_gene_id": row.gene_id,
            "seqname": row.seqname, "TF_start_1based": row.start,
            "TF_end_1based": row.end,
            "TF_strand": (genes.loc[genes.gene_id.eq(row.gene_id), "strand"].iloc[0]
                          if has_gtf else "unknown"),
            "BGC_id": row.nearest_bgc_id,
            "BGC_start_1based": row.nearest_bgc_start,
            "BGC_end_1based": row.nearest_bgc_end,
            "BGC_length_bp": row.nearest_bgc_end - row.nearest_bgc_start + 1,
            "TF_to_BGC_boundary_distance_bp": row.nearest_dist,
            "native_relation": row.gene_bgc_relation,
            "plotted_gene_count": len(genes),
            "genes_overlapping_BGC": (int(((genes.start <= row.nearest_bgc_end) &
                                           (genes.end >= row.nearest_bgc_start)).sum())
                                      if has_gtf else pd.NA),
            "GTF_available": has_gtf,
            "BGC_display_start_kb": bgc_left / 1000,
            "BGC_display_end_kb": bgc_right / 1000,
            "TF_display_start_kb": min(transform(row.start, row), transform(row.end, row)) / 1000,
            "TF_display_end_kb": max(transform(row.start, row), transform(row.end, row)) / 1000,
        })
    genes_df = pd.DataFrame(details)
    summary_df = pd.DataFrame(summaries)
    if not genes_df.empty:
        genes_df = genes_df.sort_values(["reference_species", "display_start_kb", "gene_id"])
        genes_df["display_order"] = genes_df.groupby("reference_species").cumcount() + 1
    genes_df.to_csv(OUT / "gene_order_orientation.csv", index=False)
    summary_df.to_csv(OUT / "tf_bgc_summary.csv", index=False)
    return summary_df, genes_df


def add_ortholog_groups(genes, summary, ortholog_path):
    orthologs = pd.read_excel(ortholog_path, dtype=str)
    missing = set(ORTHOLOG_COLUMNS) - set(orthologs.columns)
    if missing:
        raise ValueError(f"Required columns are missing from the ortholog table: {missing}")
    rows = []
    seen = set()
    reference_ids = [str(v).strip() for v in orthologs["Gene ID"]]
    if len(reference_ids) != len(set(reference_ids)):
        raise ValueError("Duplicate values were found in the Gene ID column")
    group_for_reference = {ref: f"G{i + 1:02d}" for i, ref in enumerate(reference_ids)}
    for i, row in orthologs.iterrows():
        source_ref = str(row["Gene ID"]).strip()
        if not source_ref or source_ref.lower() == "nan":
            continue
        for column, species in ORTHOLOG_COLUMNS.items():
            value = row[column]
            if pd.isna(value) or not str(value).strip():
                continue
            gene_id = str(value).strip()  # removes tabs in two supplied cells
            key = (species, gene_id)
            if key in seen:
                raise ValueError(f"A gene occurs in multiple ortholog groups: {key}")
            seen.add(key)
            rows.append({"ortholog_group": group_for_reference[source_ref],
                         "reference_gene": source_ref,
                         "reference_species": species, "gene_id": gene_id})
    mapping = pd.DataFrame(rows)
    if mapping.empty:
        raise ValueError("No gene IDs could be read from the ortholog table")
    local = genes[["reference_species", "gene_id", "overlaps_antiSMASH_area"]]
    validation = mapping.merge(local, on=["reference_species", "gene_id"], how="left",
                               validate="one_to_one")
    validation["status"] = np.select(
        [validation.overlaps_antiSMASH_area.eq(True).fillna(False),
         validation.overlaps_antiSMASH_area.eq(False).fillna(False)],
        ["inside_predicted_BGC", "in_plotted_flank"], default="outside_plotted_window_or_not_in_GTF")
    for idx in validation.index[validation.status.eq("outside_plotted_window_or_not_in_GTF")]:
        species = validation.at[idx, "reference_species"]
        gene_id = validation.at[idx, "gene_id"]
        all_loci = read_gtf(ROOT / GTF_BY_SPECIES[species])
        hit = all_loci[all_loci.gene_id.eq(gene_id)]
        if hit.empty:
            validation.at[idx, "status"] = "not_in_GTF"
        elif hit.iloc[0].seqname != summary.loc[summary.reference_species.eq(species), "seqname"].iloc[0]:
            validation.at[idx, "status"] = "different_scaffold"
        else:
            validation.at[idx, "status"] = "outside_plotted_window"
    validation.to_csv(OUT / "ortholog_validation.csv", index=False)
    if validation.status.ne("inside_predicted_BGC").any():
        print("WARNING: Some mapped genes are outside the predicted BGC. See ortholog_validation.csv.")
        print(validation.loc[validation.status.ne("inside_predicted_BGC"),
                             ["reference_species", "gene_id", "ortholog_group", "status"]].to_string(index=False))
    genes = genes.merge(mapping[["reference_species", "gene_id", "ortholog_group",
                                 "reference_gene"]], on=["reference_species", "gene_id"],
                        how="left", validate="one_to_one")
    genes.to_csv(OUT / "gene_order_orientation_with_orthologs.csv", index=False)
    key = mapping[["ortholog_group", "reference_gene"]].drop_duplicates()
    key = key.sort_values("ortholog_group")
    key.to_csv(OUT / "ortholog_key.csv", index=False)
    return genes, key, validation


def draw(summary, genes, key):
    plt.rcParams.update({"font.weight": "bold", "axes.labelweight": "bold",
                         "axes.titleweight": "bold", "font.size": 22,
                         "axes.labelsize": 24, "xtick.labelsize": 22,
                         "ytick.labelsize": 22})
    n = len(summary)
    palette = plt.get_cmap("tab20").colors
    group_colors = {group: palette[i % len(palette)]
                    for i, group in enumerate(key.ortholog_group)}
    # Keep G07 distinct from the red TF/candidate ortholog arrows.
    group_colors["G07"] = "#8B0000"  # Dark Red
    fig, axes = plt.subplots(n, 1, figsize=(28, 4.5 * n), sharex=True,
                             constrained_layout=True)
    x_max = float(summary.BGC_display_end_kb.max()) + FLANK_BP / 1000
    for ax, row in zip(axes, summary.itertuples(index=False)):
        ax.set_xlim(-6, x_max)
        ax.set_ylim(-0.85, 2.1)
        ax.set_yticks([])
        ax.spines[["left", "right", "top"]].set_visible(False)
        ax.axhline(0, color="#9aa3aa", lw=0.8)
        ax.add_patch(Rectangle((row.BGC_display_start_kb, -0.38),
                               row.BGC_display_end_kb - row.BGC_display_start_kb,
                               0.76, facecolor="#dcecf7", edgecolor="#5a9abf", lw=1))
        ax.annotate("", xy=(row.BGC_display_start_kb, -0.54), xytext=(0, -0.54),
                    arrowprops={"arrowstyle": "|-|", "color": "#444", "lw": 1})
        ax.text(row.BGC_display_start_kb / 2, -0.66,
                f"{row.TF_to_BGC_boundary_distance_bp:,} bp", ha="center",
                fontsize=22, fontweight="bold")
        g = genes[genes.reference_species.eq(row.reference_species)] if not genes.empty else genes
        for gene in g.itertuples(index=False):
            color = ("#FF0000" if gene.is_TF_ortholog else
                     group_colors[gene.ortholog_group] if pd.notna(gene.ortholog_group) else
                     "#b0b0b0" if gene.overlaps_antiSMASH_area else "#777777")
            if gene.display_strand == "-":
                ax.add_patch(FancyArrowPatch((gene.display_end_kb, 0),
                                             (gene.display_start_kb, 0),
                                             arrowstyle="-|>", mutation_scale=16,
                                             lw=5.0, color=color, zorder=3))
            else:
                ax.add_patch(FancyArrowPatch((gene.display_start_kb, 0),
                                             (gene.display_end_kb, 0),
                                             arrowstyle="-|>", mutation_scale=16,
                                             lw=5.0, color=color, zorder=3))
            if gene.overlaps_antiSMASH_area and not gene.is_TF_ortholog:
                ax.text((gene.display_start_kb + gene.display_end_kb) / 2,
                        0.14, f"{gene.gene_id} ({gene.ortholog_group})"
                        if pd.notna(gene.ortholog_group) else gene.gene_id,
                        rotation=45, rotation_mode="anchor", ha="left", va="bottom",
                        fontsize=20, color="#183b50", fontweight="bold")
        if not row.GTF_available:
            ax.plot([row.TF_display_start_kb, row.TF_display_end_kb], [0, 0],
                    color="#FF0000", lw=4)
            ax.text(row.BGC_display_start_kb + 0.2, -0.53, "GTF absent: gene arrows omitted",
                    fontsize=20, color="#8a4a22")
        ax.text(-0.01, 0.65, row.reference_species.replace("Aspergillus_", "A. ").replace("_", " "),
                transform=ax.transAxes, ha="right", va="center", fontsize=24,
                fontweight="bold")
        ax.text(row.TF_display_start_kb, 0.48, row.TF_gene_id,
                color="#a02b24", fontsize=22, ha="left", va="bottom",
                rotation=45, rotation_mode="anchor", fontweight="bold")
    axes[-1].set_xlabel("Position (kb) after orienting Zn2Cys6 Protein at left and BGC at right")
    for label in axes[-1].get_xticklabels():
        label.set_fontweight("bold")
    legend = [Patch(color="#FF0000", label="AO090005001236 or its candidate orthologs"),
              Patch(color="#b0b0b0", label="BGC-area genes that have no corresponding orthologs")]
    legend += [Patch(color=group_colors[r.ortholog_group],
                     label=f"{r.ortholog_group}: {r.reference_gene}")
               for r in key.itertuples(index=False)]
    # Append after the ortholog groups (below G13). Match the flank arrows.
    legend.append(Patch(color="#777777", label="Genes in the BGC-adjacent region"))
    legend_artist = fig.legend(handles=legend, loc="center left", bbox_to_anchor=(1.005, 0.5),
               frameon=False, prop={"size": 22, "weight": "bold"},
               title="Ortholog groups", title_fontsize=26)
    legend_artist.get_title().set_fontweight("bold")
    fig.savefig(OUT / "tf_bgc_neighborhoods_ortholog.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT / "tf_bgc_neighborhoods_ortholog.pdf", bbox_inches="tight")
    plt.close(fig)


def main():
    xlsx = ROOT / XLSX
    if not xlsx.exists():
        raise FileNotFoundError(f"Place {XLSX} in the input directory: {ROOT}")
    ortholog_path = ROOT / ORTHOLOG_XLSX
    if not ortholog_path.exists():
        raise FileNotFoundError(f"Place {ORTHOLOG_XLSX} in the input directory: {ROOT}")
    summary, genes = prepare(xlsx)
    genes, key, validation = add_ortholog_groups(genes, summary, ortholog_path)
    draw(summary, genes, key)
    print(summary[["reference_species", "TF_gene_id", "TF_to_BGC_boundary_distance_bp",
                   "genes_overlapping_BGC", "GTF_available"]].to_string(index=False))
    print(f"Saved to {OUT}")
    print("Gene colors: red=TF, shared colors=user-supplied ortholog groups, gray=unmatched genes.")
    print("Listed group assignments are treated as supplied; this script does not re-infer orthology.")


if __name__ == "__main__":
    main()


          reference_species     gene_id ortholog_group             status
Aspergillus_clavatus_NRRL_1 ACLA_052970            G05 different_scaffold
           reference_species     TF_gene_id  TF_to_BGC_boundary_distance_bp  genes_overlapping_BGC  GTF_available
    Aspergillus_oryzae_RIB40 AO090005001236                            6663                     14           True
 Aspergillus_flavus_NRRL3357    AFL2G_01164                            6800                     15           True
 Aspergillus_clavatus_NRRL_1    ACLA_020410                            6572                     12           True
Aspergillus_nidulans_FGSC_A4       AN0835.2                            8043                     13           True
 Aspergillus_fumigatus_Af293   AFUA_1G14945                            5485                     12           True
Aspergillus_niger_CBS_513.88  ANI_1_1770014                            7189                     11           True
 Aspergillus_terreus_NIH2624     ATEG_00855           